In [7]:
import importlib
import numpy as np
import polars as pl
import scipy.sparse as sp
import torch

from pathlib import Path

from datasets import DATASET_FOLDER, prepare_interaction_data
from util import CHECKPOINT_FOLDER, get_checkpoint_filepath, load_checkpoint, load_config_from_checkpoint

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.mps.is_available() else torch.device("cpu")

DATASET = "ml-32m-filtered"
sae_candidates = list(Path(f"{CHECKPOINT_FOLDER}/{DATASET}").glob("TopKSAE-*.ckpt"))
if len(sae_candidates) == 0:
    raise FileNotFoundError(f"No TopKSAE checkpoint found in {CHECKPOINT_FOLDER}/{DATASET}. Train ELSA + TopKSAE first.")
SAE_CHECKPOINT_PATH = str(max(sae_candidates, key=lambda p: p.stat().st_mtime))

sae_cfg = load_config_from_checkpoint(SAE_CHECKPOINT_PATH)
elsa_checkpoint_path = f"{CHECKPOINT_FOLDER}/{DATASET}/{sae_cfg['pretrained_model_checkpoint']}"
elsa_cfg = load_config_from_checkpoint(elsa_checkpoint_path)

interactions_df, train_csr, val_csr, test_csr, train_users, val_users, test_users, items = prepare_interaction_data(elsa_cfg)
train_users_to_idxs = {uid: uidx for uidx, uid in enumerate(train_users)}
val_users_to_idxs = {uid: uidx for uidx, uid in enumerate(val_users)}
test_users_to_idxs = {uid: uidx for uidx, uid in enumerate(test_users)}
items_to_idxs = {iid: iidx for iidx, iid in enumerate(items)}

items_df = pl.scan_csv(f"{DATASET_FOLDER}/movies.csv").rename({"movieId": "item_id"}).cast({"item_id": pl.String}).cast({"item_id": pl.Categorical}).collect()

elsa_model_class = getattr(importlib.import_module(elsa_cfg["model_module"]), elsa_cfg["model_class"])
elsa = elsa_model_class(train_csr.shape[1], elsa_cfg["embedding_dim"]).to(device)
_, _ = load_checkpoint(elsa, None, get_checkpoint_filepath(elsa_cfg), device, None)

sae_model_class = getattr(importlib.import_module(sae_cfg["model_module"]), sae_cfg["model_class"])
sae_extra_params = {k: sae_cfg[k] for k in sae_cfg.keys() if k in ["l1_coef", "k"]}
sae = sae_model_class(elsa_cfg["embedding_dim"], sae_cfg["embedding_dim"], sae_cfg["reconstruction_loss"], **sae_extra_params).to(device)
_, _ = load_checkpoint(sae, None, get_checkpoint_filepath(sae_cfg), device, None)

Dataset info: users=30966, items=8328, interactions=2954801
Train split info: users=24773, items=8328, interactions=2364282
Val split info: users=6193, items=8328, interactions=590519
Test split info: users=0, items=8328, interactions=0
Loaded checkpoint from /Users/vaclav.stibor/Library/CloudStorage/OneDrive-HomeCreditInternationala.s/Documents/EasyStudy/train/recsys26/checkpoints/ml-32m-filtered/ELSA-512-c2005bb7.ckpt (after 25 epochs)
Loaded checkpoint from /Users/vaclav.stibor/Library/CloudStorage/OneDrive-HomeCreditInternationala.s/Documents/EasyStudy/train/recsys26/checkpoints/ml-32m-filtered/TopKSAE-1024-4d51a427.ckpt (after 155 epochs)


In [8]:
user_id = np.random.choice(train_users)

print("Interactions:")
print(interactions_df.filter(pl.col("user_id") == user_id).join(items_df, on="item_id"))

k = 5
topk_scores, topk_idxs = elsa.recommend(torch.tensor(train_csr[train_users_to_idxs[user_id]].toarray()).to(device), k=5)
topk_scores, topk_idxs = topk_scores.flatten(), topk_idxs.flatten()
topk_item_ids = np.array([items[iidx] for iidx in topk_idxs])

print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_item_ids, "score": topk_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Interactions:
shape: (39, 5)
┌─────────┬─────────┬───────┬─────────────────────────────────┬─────────────────────────────────┐
│ user_id ┆ item_id ┆ value ┆ title                           ┆ genres                          │
│ ---     ┆ ---     ┆ ---   ┆ ---                             ┆ ---                             │
│ cat     ┆ cat     ┆ f32   ┆ str                             ┆ str                             │
╞═════════╪═════════╪═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 72641   ┆ 903     ┆ 4.0   ┆ Vertigo (1958)                  ┆ Drama|Mystery|Romance|Thriller  │
│ 72641   ┆ 904     ┆ 5.0   ┆ Rear Window (1954)              ┆ Mystery|Thriller                │
│ 72641   ┆ 908     ┆ 5.0   ┆ North by Northwest (1959)       ┆ Action|Adventure|Mystery|Roman… │
│ 72641   ┆ 953     ┆ 5.0   ┆ It's a Wonderful Life (1946)    ┆ Children|Drama|Fantasy|Romance  │
│ 72641   ┆ 1014    ┆ 5.0   ┆ Pollyanna (1960)                ┆ Children|Comedy|Drama    

In [11]:
# item_id = np.random.choice(items)
# item_id = "296"  # Pulp Fiction
item_id = "4306"  # Shrek
#item_id = "1240"

print("Item:")
print(items_df.filter(pl.col("item_id") == item_id))

k = 5
topk_scores, topk_idxs = elsa.recommend(
    torch.tensor(sp.eye(len(items), dtype=np.float32, format="csr")[items_to_idxs[item_id]].toarray()).to(device).float(), k=5
)
topk_scores, topk_idxs = topk_scores.flatten(), topk_idxs.flatten()
topk_item_ids = np.array([items[iidx] for iidx in topk_idxs])

print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_item_ids, "score": topk_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Item:
shape: (1, 3)
┌─────────┬──────────────┬─────────────────────────────────┐
│ item_id ┆ title        ┆ genres                          │
│ ---     ┆ ---          ┆ ---                             │
│ cat     ┆ str          ┆ str                             │
╞═════════╪══════════════╪═════════════════════════════════╡
│ 4306    ┆ Shrek (2001) ┆ Adventure|Animation|Children|C… │
└─────────┴──────────────┴─────────────────────────────────┘
Recommendations:
shape: (5, 4)
┌─────────┬──────────┬─────────────────────────────────┬─────────────────────────────────┐
│ item_id ┆ score    ┆ title                           ┆ genres                          │
│ ---     ┆ ---      ┆ ---                             ┆ ---                             │
│ cat     ┆ f32      ┆ str                             ┆ str                             │
╞═════════╪══════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 4886    ┆ 0.669886 ┆ Monsters, Inc. (2001)           ┆ Adventure|

In [4]:
user_idx = 123
print("Interactions")
print(interactions_df.filter(pl.col("user_id") == train_users[user_idx]).join(items_df, on="item_id"))

interaction_vector = train_csr[user_idx]  # CSR matrix, shape = (1 x num_items)
interaction_tensor = torch.tensor(interaction_vector.toarray()).to(device)  # Tensor shape = (1 x num_items)

elsa_embedding = elsa.encode(interaction_tensor)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])

sae_embedding, _, input_mean, input_std = sae.encode(elsa_embedding)  # Tensor, shape = (1 x sae_cfg["embedding_dim"]), sparse (most values are zero)

topk_neuron_values, topk_neuron_ids = torch.topk(sae_embedding, 3)
print(f"Most active neurons: {topk_neuron_ids.cpu().numpy().flatten()} with values {topk_neuron_values.detach().cpu().numpy().flatten()}")

reconstructed_elsa_embedding = sae.decode(sae_embedding, input_mean, input_std)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])
print(f"Relative reconstruction error norm: {(elsa_embedding - reconstructed_elsa_embedding).norm() / elsa_embedding.norm()}")
print(f"Cosine similarity: {torch.cosine_similarity(elsa_embedding, reconstructed_elsa_embedding).item()}")

elsa_scores = elsa.decode(elsa_embedding)
elsa_scores[interaction_tensor != 0] = 0
topk_elsa_scores, topk_elsa_idxs = torch.topk(elsa_scores, 5)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.detach().cpu().numpy().flatten(), topk_elsa_idxs.cpu().numpy().flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

reconstructed_elsa_scores = elsa.decode(reconstructed_elsa_embedding)
reconstructed_elsa_scores[interaction_tensor != 0] = 0
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = torch.topk(reconstructed_elsa_scores, 5)
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = (
    topk_reconstructed_elsa_scores.detach().cpu().numpy().flatten(),
    topk_reconstructed_elsa_idxs.cpu().numpy().flatten(),
)
topk_reconstructed_elsa_item_ids = np.array([items[iidx] for iidx in topk_reconstructed_elsa_idxs])
print("Reconstructed Recommendations:")
print(
    pl.DataFrame({"item_id": topk_reconstructed_elsa_item_ids, "score": topk_reconstructed_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Interactions
shape: (88, 5)
┌─────────┬─────────┬───────┬─────────────────────────────────┬─────────────────────────────────┐
│ user_id ┆ item_id ┆ value ┆ title                           ┆ genres                          │
│ ---     ┆ ---     ┆ ---   ┆ ---                             ┆ ---                             │
│ cat     ┆ cat     ┆ f32   ┆ str                             ┆ str                             │
╞═════════╪═════════╪═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 25221   ┆ 1       ┆ 4.0   ┆ Toy Story (1995)                ┆ Adventure|Animation|Children|C… │
│ 25221   ┆ 16      ┆ 4.0   ┆ Casino (1995)                   ┆ Crime|Drama                     │
│ 25221   ┆ 17      ┆ 5.0   ┆ Sense and Sensibility (1995)    ┆ Drama|Romance                   │
│ 25221   ┆ 21      ┆ 5.0   ┆ Get Shorty (1995)               ┆ Comedy|Crime|Thriller           │
│ 25221   ┆ 25      ┆ 4.0   ┆ Leaving Las Vegas (1995)        ┆ Drama|Romance             

/tmp/ipykernel_1008800/3905893989.py:3: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  print(interactions_df.filter(pl.col("user_id") == train_users[user_idx]).join(items_df, on="item_id"))


Recommendations:
shape: (5, 4)
┌─────────┬───────────┬─────────────────────────────────┬───────────────────────┐
│ item_id ┆ score     ┆ title                           ┆ genres                │
│ ---     ┆ ---       ┆ ---                             ┆ ---                   │
│ cat     ┆ f32       ┆ str                             ┆ str                   │
╞═════════╪═══════════╪═════════════════════════════════╪═══════════════════════╡
│ 36      ┆ 10.088968 ┆ Dead Man Walking (1995)         ┆ Crime|Drama           │
│ 1784    ┆ 7.958158  ┆ As Good as It Gets (1997)       ┆ Comedy|Drama|Romance  │
│ 265     ┆ 7.949216  ┆ Like Water for Chocolate (Como… ┆ Drama|Fantasy|Romance │
│ 1041    ┆ 7.037684  ┆ Secrets & Lies (1996)           ┆ Drama                 │
│ 357     ┆ 6.745351  ┆ Four Weddings and a Funeral (1… ┆ Comedy|Romance        │
└─────────┴───────────┴─────────────────────────────────┴───────────────────────┘
Reconstructed Recommendations:
shape: (5, 4)
┌─────────┬──────────┬

/tmp/ipykernel_1008800/3905893989.py:27: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_1008800/3905893989.py:42: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")


In [5]:
item_id = "4306"  # Shrek
interaction_tensor = (
    torch.tensor(torch.tensor(sp.eye(len(items), dtype=np.float32, format="csr")[items_to_idxs[item_id]].toarray())).to(device).float()
)  # Tensor shape = (1 x num_items)

elsa_embedding = elsa.encode(interaction_tensor)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])

sae_embedding, _, input_mean, input_std = sae.encode(elsa_embedding)  # Tensor, shape = (1 x sae_cfg["embedding_dim"]), sparse (most values are zero)

topk_neuron_values, topk_neuron_ids = torch.topk(sae_embedding, 3)
print(f"Most active neurons: {topk_neuron_ids.cpu().numpy().flatten()} with values {topk_neuron_values.detach().cpu().numpy().flatten()}")

reconstructed_elsa_embedding = sae.decode(sae_embedding, input_mean, input_std)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])
print(f"Relative reconstruction error norm: {(elsa_embedding - reconstructed_elsa_embedding).norm() / elsa_embedding.norm()}")
print(f"Cosine similarity: {torch.cosine_similarity(elsa_embedding, reconstructed_elsa_embedding).item()}")

elsa_scores = elsa.decode(elsa_embedding)
elsa_scores[interaction_tensor != 0] = 0
topk_elsa_scores, topk_elsa_idxs = torch.topk(elsa_scores, 5)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.detach().cpu().numpy().flatten(), topk_elsa_idxs.cpu().numpy().flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

reconstructed_elsa_scores = elsa.decode(reconstructed_elsa_embedding)
reconstructed_elsa_scores[interaction_tensor != 0] = 0
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = torch.topk(reconstructed_elsa_scores, 5)
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = (
    topk_reconstructed_elsa_scores.detach().cpu().numpy().flatten(),
    topk_reconstructed_elsa_idxs.cpu().numpy().flatten(),
)
topk_reconstructed_elsa_item_ids = np.array([items[iidx] for iidx in topk_reconstructed_elsa_idxs])
print("Reconstructed Recommendations:")
print(
    pl.DataFrame({"item_id": topk_reconstructed_elsa_item_ids, "score": topk_reconstructed_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Most active neurons: [4222 3912 1657] with values [4.528693  2.004158  0.7210276]
Relative reconstruction error norm: 0.9933446645736694
Cosine similarity: 0.9484307169914246
Recommendations:
shape: (5, 4)
┌─────────┬───────────┬─────────────────────────────────┬─────────────────────────────────┐
│ item_id ┆ score     ┆ title                           ┆ genres                          │
│ ---     ┆ ---       ┆ ---                             ┆ ---                             │
│ cat     ┆ f32       ┆ str                             ┆ str                             │
╞═════════╪═══════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 8360    ┆ 21.489857 ┆ Shrek 2 (2004)                  ┆ Adventure|Animation|Children|C… │
│ 4886    ┆ 18.588537 ┆ Monsters, Inc. (2001)           ┆ Adventure|Animation|Children|C… │
│ 6377    ┆ 16.577394 ┆ Finding Nemo (2003)             ┆ Adventure|Animation|Children|C… │
│ 6539    ┆ 12.066116 ┆ Pirates of the Caribbean: The … ┆ 

/tmp/ipykernel_1008800/1794339784.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(torch.tensor(sp.eye(len(items), dtype=np.float32, format="csr")[items_to_idxs[item_id]].toarray())).to(device).float()
/tmp/ipykernel_1008800/1794339784.py:25: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_1008800/1794339784.py:40: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")


In [6]:
user_idx = 123
print("Interactions")
print(interactions_df.filter(pl.col("user_id") == train_users[user_idx]).join(items_df, on="item_id"))

interaction_vector = train_csr[user_idx]  # CSR matrix, shape = (1 x num_items)
interaction_tensor = torch.tensor(interaction_vector.toarray()).to(device)  # Tensor shape = (1 x num_items)

elsa_embedding = elsa.encode(interaction_tensor)  # Tensor, shape = (1 x elsa_cfg["embedding_dim"])

sae_embedding, _, input_mean, input_std = sae.encode(elsa_embedding)  # Tensor, shape = (1 x sae_cfg["embedding_dim"]), sparse (most values are zero)

boosted_sae_embedding = sae_embedding.clone()
boosted_sae_embedding[0, 4671] += 50
# boosted_sae_embedding[0, 3003] += 10
reconstructed_elsa_embedding = sae.decode(boosted_sae_embedding, input_mean, input_std)
print(f"Relative reconstruction error norm: {(elsa_embedding - reconstructed_elsa_embedding).norm() / elsa_embedding.norm()}")

elsa_scores = elsa.decode(elsa_embedding)
elsa_scores[interaction_tensor != 0] = 0
topk_elsa_scores, topk_elsa_idxs = torch.topk(elsa_scores, 5)
topk_elsa_scores, topk_elsa_idxs = topk_elsa_scores.detach().cpu().numpy().flatten(), topk_elsa_idxs.cpu().numpy().flatten()
topk_elsa_item_ids = np.array([items[iidx] for iidx in topk_elsa_idxs])
print("Recommendations:")
print(
    pl.DataFrame({"item_id": topk_elsa_item_ids, "score": topk_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

reconstructed_elsa_scores = elsa.decode(reconstructed_elsa_embedding)
reconstructed_elsa_scores[interaction_tensor != 0] = 0
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = torch.topk(reconstructed_elsa_scores, 5)
topk_reconstructed_elsa_scores, topk_reconstructed_elsa_idxs = (
    topk_reconstructed_elsa_scores.detach().cpu().numpy().flatten(),
    topk_reconstructed_elsa_idxs.cpu().numpy().flatten(),
)
topk_reconstructed_elsa_item_ids = np.array([items[iidx] for iidx in topk_reconstructed_elsa_idxs])
print("Reconstructed Recommendations:")
print(
    pl.DataFrame({"item_id": topk_reconstructed_elsa_item_ids, "score": topk_reconstructed_elsa_scores}, schema_overrides={"item_id": pl.Categorical})
    .join(items_df, on="item_id")
    .sort(by="score", descending=True)
)

Interactions
shape: (88, 5)
┌─────────┬─────────┬───────┬─────────────────────────────────┬─────────────────────────────────┐
│ user_id ┆ item_id ┆ value ┆ title                           ┆ genres                          │
│ ---     ┆ ---     ┆ ---   ┆ ---                             ┆ ---                             │
│ cat     ┆ cat     ┆ f32   ┆ str                             ┆ str                             │
╞═════════╪═════════╪═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 25221   ┆ 1       ┆ 4.0   ┆ Toy Story (1995)                ┆ Adventure|Animation|Children|C… │
│ 25221   ┆ 16      ┆ 4.0   ┆ Casino (1995)                   ┆ Crime|Drama                     │
│ 25221   ┆ 17      ┆ 5.0   ┆ Sense and Sensibility (1995)    ┆ Drama|Romance                   │
│ 25221   ┆ 21      ┆ 5.0   ┆ Get Shorty (1995)               ┆ Comedy|Crime|Thriller           │
│ 25221   ┆ 25      ┆ 4.0   ┆ Leaving Las Vegas (1995)        ┆ Drama|Romance             

/tmp/ipykernel_1008800/1355882819.py:3: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  print(interactions_df.filter(pl.col("user_id") == train_users[user_idx]).join(items_df, on="item_id"))
/tmp/ipykernel_1008800/1355882819.py:26: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
/tmp/ipykernel_1008800/1355882819.py:41: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  .join(items_df, on="item_id")
